# Aurum Market · motor de descubrimiento y control de catálogo

**Asignatura:** Bases de Datos Vectoriales — Máster en IA y Cloud Computing
**Entregable:** búsqueda semántica de productos + detección de altas duplicadas

## El problema

Aurum Market tiene un catálogo de 15.000 productos aportados por vendedores distintos. La búsqueda
actual solo encuentra coincidencias literales de palabras, así que una persona que describe lo que
necesita sin usar las palabras exactas del título no encuentra nada útil. Además, algunas fichas se
publican varias veces porque llegan con el título reordenado, la marca omitida o la descripción
ligeramente distinta.

Este notebook construye una primera versión del motor que resuelve dos tareas:

1. **Descubrimiento semántico**: dada una consulta en lenguaje natural, devolver un top-k de productos
   ordenado por relevancia, con la opción de filtrar por marca.
2. **Control de altas**: dada una ficha nueva, encontrar el producto más parecido del catálogo y decidir
   si es un duplicado que debería revisarse antes de publicarse.

**Fuera de alcance (explícito en el enunciado):** no se construye un RAG, no se genera texto y no se usa
un LLM para resolver, etiquetar o reordenar consultas. El sistema se apoya en embeddings, un índice
vectorial y reglas medibles, no en generación de lenguaje.

## Cómo está organizado el notebook

El recorrido sigue el mismo orden en el que se tomaron las decisiones, de modo que cada resultado se
apoya en el anterior:

1. Configuración del entorno
2. Carga de datos
3. Representación vectorial: baseline léxico y comparación de dos configuraciones de embeddings
4. Embeddings del catálogo completo
5. Base de datos vectorial: esquema, índice ANN e ingesta idempotente
6. Recuperación: búsqueda global y búsqueda filtrada por marca
7. Evaluación de calidad del ranking (nDCG, Recall, MRR)
8. Fidelidad del ANN frente a un oráculo exacto
9. Atribución de errores (con los casos anteriores todavía frescos)
10. Latencia
11. Mutaciones del catálogo (altas, bajas y actualizaciones)
12. Detección de altas duplicadas
13. Artefactos de entrega (`resultados_busqueda.csv`, `resultados_duplicados.csv`, `metricas_desarrollo.json`)
14. Limpieza
15. Conclusión y decisión recomendada

Cada bloque de código va precedido de una celda de texto que explica **por qué** se toma esa decisión,
no solo qué hace el código. El objetivo declarado en el enunciado es justamente ese: entender cuándo los
resultados de una base vectorial son útiles y qué capa explica sus fallos, no solo conseguir que algo se
ejecute.

## 1. Configuración del entorno

### Decisión: motor vectorial

Se utiliza **Chroma** en su modo local persistente (`PersistentClient`), sin necesidad de Docker ni de
ningún servicio externo. Se eligió por tres motivos, coherentes con que el resultado sea sencillo de
ejecutar y de entender:

- Fácil instalación.
- Persiste en disco.
- Expone directamente su SDK nativo.

### Decisión: modelo de embeddings

Se usa `intfloat/multilingual-e5-small` (384 dimensiones, ejecutable en CPU) porque el catálogo está en
español y la familia E5 está entrenada específicamente para tareas de recuperación con aprendizaje
contrastivo. E5 distingue explícitamente el papel de cada texto anteponiendo `query:` a las consultas y
`passage:` a los documentos — no es una etiqueta decorativa, forma parte de la distribución con la que se
entrenó el modelo, así que se respeta en todo el notebook.

In [ ]:
from pathlib import Path
import json
import time

import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer
import chromadb

pd.set_option("display.max_colwidth", 80)

DATA_DIR = Path("datos")
CACHE_DIR = Path("cache")
CHROMA_DIR = Path("chroma_db")
RESULTS_DIR = Path("resultados")
for d in (CACHE_DIR, RESULTS_DIR):
    d.mkdir(exist_ok=True)

TOP_K = 10
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

# Relevancia graduada tal como la define el enunciado: E=3, S=2, C=1, I=0.
ESCI_GAINS = {"E": 3, "S": 2, "C": 1, "I": 0}
RELEVANT_LABELS = {"E", "S"}

print("Entorno listo.")

Entorno listo.


## 2. Carga de datos

Se trabaja directamente sobre el **catálogo completo** (`catalogo_productos.csv.gz`, 15.000 productos),
que es el recorrido que se evalúa según el enunciado. `catalogo_muestra.csv` (1.500 filas con el mismo
esquema) se usó por separado, más adelante, para decidir con rapidez qué representación textual
utilizar antes de asumir el coste de codificar el catálogo completo.

Además `brand`, `color` y `text` pueden venir vacíos. El enunciado es explícito: *"los valores vacíos son
información ausente, no la cadena literal `nan`"*. Por eso se rellenan con cadena vacía (`""`) en el
mismo punto de carga, de forma que todo el pipeline (embeddings, metadatos de Chroma, filtros) vea
siempre el mismo tratamiento y nunca la palabra `"nan"` escrita como texto.

In [2]:
catalog = pd.read_csv(DATA_DIR / "catalogo_productos.csv.gz")
catalog = catalog.fillna({"title": "", "brand": "", "color": "", "text": "", "locale": ""})

dev_queries = pd.read_csv(DATA_DIR / "consultas_desarrollo.csv")
dev_qrels = pd.read_csv(DATA_DIR / "relevancias_desarrollo.csv")
eval_queries = pd.read_csv(DATA_DIR / "consultas_evaluacion.csv")
filtered_queries = pd.read_csv(DATA_DIR / "consultas_filtradas.csv")
catalog_events = pd.read_csv(DATA_DIR / "eventos_catalogo.csv").sort_values("sequence")
dev_altas = pd.read_csv(DATA_DIR / "altas_desarrollo.csv")
eval_altas = pd.read_csv(DATA_DIR / "altas_evaluacion.csv")

print(f"Catálogo completo: {len(catalog):,} productos, {catalog['brand'].nunique():,} marcas distintas")
print(f"Consultas de desarrollo: {len(dev_queries)} | juicios de relevancia: {len(dev_qrels)}")
print(f"Consultas de evaluación (ciegas): {len(eval_queries)}")
print(f"Consultas filtradas por marca: {len(filtered_queries)}")
print(f"Eventos de catálogo: {len(catalog_events)} | altas dev: {len(dev_altas)} | altas eval: {len(eval_altas)}")
catalog.head(3)

Catálogo completo: 15,000 productos, 9,055 marcas distintas
Consultas de desarrollo: 8 | juicios de relevancia: 248
Consultas de evaluación (ciegas): 12
Consultas filtradas por marca: 4
Eventos de catálogo: 24 | altas dev: 14 | altas eval: 14


,record_id,product_id,title,brand,color,locale,text,catalog_version,active
0,e1a0e559-6a49-5be5-b617-ec8a4899e975,B000G3T55M,"NIKE Legasee Legging Swoosh Pantalones Deportivos, Mujer, Negro (Black/White...",NIKE,Negro (Black/White 011),es,"NIKE Legasee Legging Swoosh Pantalones Deportivos, Mujer, Negro (Black/White...",1,True
1,0df8a596-0bc2-5deb-bc84-69b63972f975,B07NV4L2W5,"Interruptor Universal Inteligente con Wi-Fi, con Control Remoto Meross App. ...",meross,Blanco,es,"Interruptor Universal Inteligente con Wi-Fi, con Control Remoto Meross App. ...",1,True
2,ef061958-504a-505c-a7c8-0433be2d7630,B01BYFSX6M,"TECKNET Mini Ratón Inalámbrico Wireless Mouse Óptico, Omni 2.4G Ratón Portát...",TECKNET,Azul,es,"TECKNET Mini Ratón Inalámbrico Wireless Mouse Óptico, Omni 2.4G Ratón Portát...",1,True


## 3. Representación vectorial y baseline

Se decide qué texto codificar y con qué modelo, evaluando sobre una muestra de 1.500 productos (suficiente porque contiene todos los relevantes de las 8 consultas de desarrollo) para no pagar el coste de codificar los 15.000 productos varias veces.

Se comparan tres sistemas:

TF-IDF + coseno sobre text — baseline léxico interpretable, referencia mínima a superar.
E5 (título) — codifica solo el título, texto corto y limpio.
E5 (texto completo) — codifica el campo text ya enriquecido con marca, color, características y descripción.
La comparación A vs. B no es solo entre modelos, sino entre cuánta información del producto ve el encoder, midiendo su impacto en las tres métricas de evaluación.

In [3]:
sample_catalog = pd.read_csv(DATA_DIR / "catalogo_muestra.csv")
sample_catalog = sample_catalog.fillna({"title": "", "brand": "", "color": "", "text": ""})

model = SentenceTransformer("intfloat/multilingual-e5-small")

def encode_passages(texts, batch_size=64, show_progress_bar=False):
    """Codifica textos de producto anteponiendo el prefijo `passage:` que espera E5."""
    prefixed = ["passage: " + t for t in texts]
    return model.encode(prefixed, batch_size=batch_size, normalize_embeddings=True,
                         show_progress_bar=show_progress_bar)

def encode_queries(texts, batch_size=32):
    """Codifica consultas anteponiendo el prefijo `query:` que espera E5."""
    prefixed = ["query: " + t for t in texts]
    return model.encode(prefixed, batch_size=batch_size, normalize_embeddings=True,
                         show_progress_bar=False)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [4]:
def ndcg_at_k(ranked_product_ids, judged_labels, k=10):
    """nDCG@k con el mapeo de ganancia E=3, S=2, C=1, I=0 (el que fija el enunciado)."""
    gains = [ESCI_GAINS.get(judged_labels.get(pid), 0) for pid in ranked_product_ids[:k]]
    discounts = np.log2(np.arange(2, len(gains) + 2))
    dcg = float(np.sum(np.array(gains) / discounts))
    ideal_labels = sorted(judged_labels.values(), key=lambda l: -ESCI_GAINS[l])
    ideal_gains = [ESCI_GAINS[l] for l in ideal_labels[:k]]
    idiscounts = np.log2(np.arange(2, len(ideal_gains) + 2))
    idcg = float(np.sum(np.array(ideal_gains) / idiscounts)) if ideal_gains else 0.0
    return dcg / idcg if idcg > 0 else 0.0


def recall_at_k(ranked_product_ids, judged_labels, k=10):
    """Recall@k: proporción de productos relevantes (E o S) que aparecen en el top-k."""
    relevant = {pid for pid, label in judged_labels.items() if label in RELEVANT_LABELS}
    if not relevant:
        return 0.0
    return len(set(ranked_product_ids[:k]) & relevant) / len(relevant)


def mrr_at_k(ranked_product_ids, judged_labels, k=10):
    """MRR@k: inverso de la posición del primer resultado relevante (0 si no aparece)."""
    relevant = {pid for pid, label in judged_labels.items() if label in RELEVANT_LABELS}
    for position, pid in enumerate(ranked_product_ids[:k], start=1):
        if pid in relevant:
            return 1.0 / position
    return 0.0


def evaluate_dev_queries(score_fn, label):
    """Aplica score_fn (texto de consulta -> array de scores) a las 8 consultas de desarrollo."""
    rows = []
    for _, query_row in dev_queries.iterrows():
        judged = (
            dev_qrels[dev_qrels["query_id"] == query_row["query_id"]]
            .set_index("product_id")["esci_label"].to_dict()
        )
        scores = score_fn(query_row["query_text"])
        ranking = sample_catalog["product_id"].to_numpy()[np.argsort(-scores)]
        rows.append({
            "query_id": query_row["query_id"],
            "ndcg@10": ndcg_at_k(ranking, judged),
            "recall@10": recall_at_k(ranking, judged),
            "mrr@10": mrr_at_k(ranking, judged),
        })
    result = pd.DataFrame(rows)
    summary = result[["ndcg@10", "recall@10", "mrr@10"]].mean()
    summary.name = label
    return summary

In [5]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

tfidf_vectorizer = TfidfVectorizer()
tfidf_matrix = tfidf_vectorizer.fit_transform(sample_catalog["text"])

def tfidf_score(query_text):
    query_vector = tfidf_vectorizer.transform([query_text])
    return cosine_similarity(query_vector, tfidf_matrix).ravel()

baseline_summary = evaluate_dev_queries(tfidf_score, "TF-IDF (baseline léxico, campo text)")
baseline_summary

ndcg@10      0.520467
recall@10    0.194679
mrr@10       0.750000
Name: TF-IDF (baseline léxico, campo text), dtype: float64

In [6]:
title_embeddings = encode_passages(sample_catalog["title"].tolist())
text_embeddings = encode_passages(sample_catalog["text"].tolist())

_query_cache = {}
def _cached_query_vector(query_text):
    if query_text not in _query_cache:
        _query_cache[query_text] = encode_queries([query_text])[0]
    return _query_cache[query_text]

config_a_summary = evaluate_dev_queries(
    lambda q: title_embeddings @ _cached_query_vector(q), "E5 · config A (solo title)"
)
config_b_summary = evaluate_dev_queries(
    lambda q: text_embeddings @ _cached_query_vector(q), "E5 · config B (campo text completo)"
)

comparison = pd.DataFrame([baseline_summary, config_a_summary, config_b_summary])
comparison

,ndcg@10,recall@10,mrr@10
"TF-IDF (baseline léxico, campo text)",0.520467,0.194679,0.7500
E5 · config A (solo title),0.622960,0.282574,0.7750
E5 · config B (campo text completo),0.676594,0.283024,0.8125


### Decisión de representación

Los resultados se interpretan en dos comparaciones separadas, no solo mirando quién saca mejor nota:

¿Merece la pena lo denso frente a lo léxico? Solo tiene sentido usar embeddings y una base vectorial si E5 supera claramente a TF-IDF. En Aurum Market esto se cumple: las consultas describen una necesidad ("herramienta inalámbrica para perforar") sin usar el término exacto del catálogo ("taladro"), un caso donde el matching por palabras falla y el modelo denso sí aporta.
¿Título o texto completo? El título es más limpio pero más pobre; el campo text completo añade marca, color y características a costa de más ruido. Gana la opción que sea mejor en las tres métricas a la vez, y esa elección se fija de antemano para no manipular la comparación a posteriori.
Resultado: se elige E5 sobre el campo text completo, y con esa configuración se codifican tanto el catálogo como las consultas en el resto del notebook.

### Normalización y significado del score

Los embeddings se normalizan (norma L2 = 1) porque E5 fue entrenado para usarse con similitud coseno, y con vectores normalizados el producto escalar equivale exactamente a esa similitud — usar vectores sin normalizar daría un comportamiento distinto al que el modelo aprendió.

Chroma, configurado con space="cosine", no devuelve la similitud sino su complemento, la distancia (1 - similitud, donde menor es mejor). Para evitar confundir ambas convenciones, cada score que sale de Chroma se convierte siempre a similitud (score = 1 - distancia) antes de usarlo en cualquier comparación.

## 4. Embeddings del catálogo completo

Codificar 15.000 textos con E5 en CPU tarda varios minutos (en esta máquina, unos 13-15). Es un coste que
se paga una sola vez: los vectores se guardan en `cache/catalog_embeddings.npy` junto con los
`record_id` en el mismo orden, y si el notebook se vuelve a ejecutar se reutiliza la caché en lugar de
recalcular. La comprobación de que los IDs guardados coinciden exactamente con los del catálogo actual
evita el error silencioso de servir una caché desalineada tras cualquier cambio en los datos de entrada.

In [7]:
EMBEDDINGS_CACHE = CACHE_DIR / "catalog_embeddings.npy"
RECORD_IDS_CACHE = CACHE_DIR / "catalog_record_ids.npy"

if EMBEDDINGS_CACHE.exists() and RECORD_IDS_CACHE.exists():
    catalog_embeddings = np.load(EMBEDDINGS_CACHE)
    cached_ids = np.load(RECORD_IDS_CACHE, allow_pickle=True)
    assert list(cached_ids) == list(catalog["record_id"]), "La caché no coincide con el catálogo actual"
    print(f"Embeddings cargados desde caché: {catalog_embeddings.shape}")
else:
    start = time.perf_counter()
    catalog_embeddings = encode_passages(catalog["text"].tolist(), show_progress_bar=True)
    print(f"Codificación del catálogo completo: {time.perf_counter() - start:.1f} s")
    np.save(EMBEDDINGS_CACHE, catalog_embeddings)
    np.save(RECORD_IDS_CACHE, catalog["record_id"].to_numpy())

assert catalog_embeddings.shape == (len(catalog), 384)

Embeddings cargados desde caché: (15000, 384)


## 5. Base de datos vectorial

### Índice ANN

Chroma Single Node solo ofrece **HNSW** como familia de índice ANN (no se puede elegir IVF u otras, a
diferencia de otros motores vistos en clase). Lo que sí se controla explícitamente son sus parámetros,
fijados aquí para que el experimento sea reproducible:

- `space="cosine"`: coherente con la normalización L2 decidida en la sección 3.
- `max_neighbors=24` (M): número de conexiones por nodo del grafo.
- `ef_construction=120`: amplitud de búsqueda al construir el grafo (más alto → grafo más preciso, ingesta
  más lenta).
- `ef_search=128`: amplitud de búsqueda en tiempo de consulta (más alto → más recall, más latencia).

Estos parámetros solo pueden fijarse al crear la colección; cambiarlos después implica borrar y
reconstruir el índice desde cero. Por eso la celda comprueba la configuración real de la colección tras
crearla, en vez de asumir que se aplicó.

In [8]:
client = chromadb.PersistentClient(path=str(CHROMA_DIR))

HNSW_CONFIG = {"space": "cosine", "max_neighbors": 24, "ef_construction": 120, "ef_search": 128}
COLLECTION_NAME = "aurum_market_catalogo"

collection = client.get_or_create_collection(
    name=COLLECTION_NAME,
    embedding_function=None,   # los embeddings los calculamos nosotros; Chroma no debe inferir los suyos
    configuration={"hnsw": HNSW_CONFIG},
    metadata={"proyecto": "aurum-market", "modelo": "intfloat/multilingual-e5-small"},
)

actual_hnsw = collection.configuration["hnsw"]
for key, expected in HNSW_CONFIG.items():
    assert actual_hnsw[key] == expected, f"{key}: esperado {expected}, real {actual_hnsw[key]}"
print("Configuración HNSW verificada:", {k: actual_hnsw[k] for k in HNSW_CONFIG})

Configuración HNSW verificada: {'space': 'cosine', 'max_neighbors': 24, 'ef_construction': 120, 'ef_search': 128}


### Ingesta por lotes e idempotente

Se ingiere en lotes de 500 registros (un tamaño razonable para no saturar memoria ni hacer demasiadas
peticiones; no pretende ser un valor óptimo universal). La operación es `upsert`, no `insert`: como el
`record_id` es determinista, volver a ejecutar la ingesta sobre los mismos datos actualiza los registros
existentes en vez de duplicarlos. Esa propiedad es la que permite reanudar la carga tras una interrupción
sin miedo a inflar el recuento — se comprueba explícitamente repitiendo la ingesta completa una segunda
vez y verificando que el recuento no cambia.

In [9]:
def row_to_metadata(row):
    return {
        "product_id": str(row["product_id"]),
        "title": str(row["title"]),
        "brand": str(row["brand"]),
        "color": str(row["color"]),
        "locale": str(row["locale"]),
        "catalog_version": int(row["catalog_version"]),
        "active": bool(row["active"]),
    }

def ingest_catalog(df, embeddings, batch_size=500):
    for start in range(0, len(df), batch_size):
        chunk = df.iloc[start:start + batch_size]
        collection.upsert(
            ids=chunk["record_id"].tolist(),
            embeddings=embeddings[start:start + batch_size].tolist(),
            documents=chunk["text"].tolist(),
            metadatas=[row_to_metadata(row) for _, row in chunk.iterrows()],
        )

ingestion_started = time.perf_counter()
ingest_catalog(catalog, catalog_embeddings)
ingestion_ms = (time.perf_counter() - ingestion_started) * 1000

record_count = collection.count()
print(f"Ingesta: {ingestion_ms:.0f} ms · registros visibles: {record_count:,}")
assert record_count == len(catalog), "El recuento no coincide con el catálogo cargado"

Ingesta: 102638 ms · registros visibles: 15,000


In [10]:
# Verificación de idempotencia: repetir la ingesta completa no debe aumentar el recuento.
ingest_catalog(catalog, catalog_embeddings)
record_count_repeat = collection.count()
print(f"Registros tras repetir la ingesta completa: {record_count_repeat:,}")
assert record_count_repeat == record_count, "La ingesta no es idempotente"

Registros tras repetir la ingesta completa: 15,000


## 6. Recuperación

Se expone una única función `search()` que actúa como interfaz común del sistema: recibe una consulta en
texto plano y devuelve una lista de resultados normalizados con `product_id`, `rank`, `title`, `brand` y
`score`. Internamente hace tres cosas, siempre en este orden:

1. Codifica la consulta con el prefijo `query:` (nunca `passage:` — una consulta corta y una ficha de
   producto no cumplen el mismo papel para E5).
2. Envía el filtro de marca, si lo hay, **como parte de la propia consulta a Chroma** (`where={...}`), no
   como un filtrado posterior en Python. Esto importa: filtrar después de recuperar el top-k global podría
   perder candidatos válidos que quedaron fuera de esos primeros k resultados pero sí cumplen la marca.
3. Convierte la distancia coseno nativa de Chroma en similitud (`1 - distancia`) antes de devolverla, para
   no mezclar convenciones de score en el resto del notebook.

También se contemplan los casos límite que pide el enunciado: una colección vacía o un filtro sin
resultados devuelven una lista vacía en lugar de lanzar una excepción.

In [11]:
def search(query_text, top_k=10, brand=None):
    """Busca los top_k productos más parecidos a query_text. Si brand no es None, el filtro
    se aplica dentro de la propia consulta vectorial, no después de recuperar los resultados.
    Devuelve una lista de dicts normalizados; una lista vacía si no hay resultados."""
    if collection.count() == 0:
        return []
    query_vector = encode_queries([query_text])[0]
    response = collection.query(
        query_embeddings=[query_vector.tolist()],
        n_results=top_k,
        where={"brand": brand} if brand else None,
        include=["metadatas", "distances"],
    )
    if not response["ids"][0]:
        return []
    hits = []
    for rank, (record_id, metadata, distance) in enumerate(
        zip(response["ids"][0], response["metadatas"][0], response["distances"][0]), start=1
    ):
        hits.append({
            "record_id": record_id,
            "product_id": metadata["product_id"],
            "rank": rank,
            "title": metadata["title"],
            "brand": metadata["brand"],
            "score": 1.0 - distance,
        })
    return hits


# Caso límite 1: un filtro que no puede cumplir ningún producto real -> lista vacía, no un error.
assert search("cualquier cosa", brand="MARCA-QUE-NO-EXISTE-XYZ") == []

# Caso límite 2: una colección vacía (por ejemplo, antes de la primera ingesta) -> lista vacía.
_empty_collection_name = "prueba_coleccion_vacia"
_previous_collection = collection
collection = client.get_or_create_collection(name=_empty_collection_name, embedding_function=None)
assert search("cualquier consulta") == []
collection = _previous_collection
client.delete_collection(_empty_collection_name)

print("Casos límite comprobados: filtro sin resultados y colección vacía devuelven [] sin lanzar excepción.")

Casos límite comprobados: filtro sin resultados y colección vacía devuelven [] sin lanzar excepción.


In [12]:
pd.DataFrame(search("taladro inalámbrico para perforar", top_k=5))

,record_id,product_id,rank,title,brand,score
0,e29550b9-3563-5c5f-9835-9d808c885a0d,B07GSD93Q8,1,"Einhell Taladro de impacto sin cable TE-CD 12/1 Li-i (2x 2,0Ah Li-Ion, 12 V,...",Einhell,0.901929
1,2a5aa063-7d33-5243-9ec2-3687ace89a66,B01A5VQHBY,2,Taladro Percutor Brushless 20V Worx WX373,WORX,0.894192
2,39885d35-db49-56d4-b493-fa768101f6fd,B0071T3MOO,3,"Bosch 12V System GSB 12V-15 - Taladro Percutor a Batería, 30 Nm, 1300 Rpm, s...",Bosch Professional,0.893274
3,4d9749bc-d9b7-523c-bfa6-bab0cd877737,B09874QBSH,4,Adaptador de mandril de taladro Convertidor sin llave 0.3-3.6mm 0.3-6.5mm co...,bulingbuling,0.893184
4,d07a2de7-bd29-5808-978a-985a9d9624a5,B07J5CK31F,5,"Taladro Percutor, Meterk 850W Taladro Eléctrico de 3000 RPM, Martillo Taladr...",Meterk,0.893104


### Búsqueda filtrada por marca

`consultas_filtradas.csv` define cuatro consultas, cada una con una marca obligatoria. Se comprueba, para
las cuatro, que **todos** los resultados devueltos pertenecen a esa marca — si el filtro se hubiera
aplicado mal (por ejemplo, recuperando el top-10 global y descartando después), esta comprobación lo
detectaría inmediatamente.

In [13]:
filtered_results = []
for _, filter_row in filtered_queries.iterrows():
    hits = search(filter_row["query_text"], top_k=10, brand=filter_row["filter_value"])
    all_match_brand = len(hits) > 0 and all(h["brand"] == filter_row["filter_value"] for h in hits)
    filtered_results.append({
        "workload_id": filter_row["workload_id"],
        "marca_exigida": filter_row["filter_value"],
        "n_resultados": len(hits),
        "todos_cumplen_marca": all_match_brand,
    })
    assert all_match_brand, f"El filtro de marca falló en {filter_row['workload_id']}"

pd.DataFrame(filtered_results)

,workload_id,marca_exigida,n_resultados,todos_cumplen_marca
0,FILTER-001,Einhell,10,True
1,FILTER-002,Apple,10,True
2,FILTER-003,NIKE,10,True
3,FILTER-004,SAMSUNG,10,True


## 7. Calidad del ranking sobre el catálogo completo

Se repiten las mismas tres métricas de la sección 3 (nDCG@10, Recall@10, MRR@10), pero ahora:

- sobre el **catálogo completo** de 15.000 productos ingerido en Chroma (antes se usó la muestra de 1.500
  solo para elegir representación);
- pasando por la **base de datos vectorial real** (`search()`, que consulta el índice HNSW), no por un
  cálculo de fuerza bruta en NumPy.

Esta es la referencia de calidad de todo el sistema. Las funciones `ndcg_at_k`, `recall_at_k` y
`mrr_at_k` ya se definieron en la sección 3 y se reutilizan sin cambios, precisamente para no introducir
una definición distinta de "relevante" a mitad de camino.

In [14]:
dev_ranking_rows = []
for _, query_row in dev_queries.iterrows():
    judged = (
        dev_qrels[dev_qrels["query_id"] == query_row["query_id"]]
        .set_index("product_id")["esci_label"].to_dict()
    )
    hits = search(query_row["query_text"], top_k=TOP_K)
    ranked_product_ids = [h["product_id"] for h in hits]
    dev_ranking_rows.append({
        "query_id": query_row["query_id"],
        "query_text": query_row["query_text"],
        "ndcg@10": ndcg_at_k(ranked_product_ids, judged),
        "recall@10": recall_at_k(ranked_product_ids, judged),
        "mrr@10": mrr_at_k(ranked_product_ids, judged),
    })

dev_ranking_df = pd.DataFrame(dev_ranking_rows)
dev_metrics = dev_ranking_df[["ndcg@10", "recall@10", "mrr@10"]].mean().to_dict()
print({k: round(v, 4) for k, v in dev_metrics.items()})
dev_ranking_df

{'ndcg@10': 0.5333, 'recall@10': 0.2051, 'mrr@10': 0.7125}


,query_id,query_text,ndcg@10,recall@10,mrr@10
0,13357,base tapizada 160x200 sin patas,0.627829,0.225806,1.0
1,18868,botines marrones mujer tacon medio,0.314709,0.333333,0.5
2,28703,convertibles 2 en 1 portátil tactil,0.835780,0.205128,1.0
3,31224,cámaras bridge baratas,0.437192,0.200000,1.0
4,33633,disfraz halloween talla grande hombre,0.000000,0.000000,0.0
5,38249,estantes sin taladro habitacion,0.148764,0.057143,0.2
6,43240,funda ipad air 4 sin tapa,0.902181,0.285714,1.0
7,61533,lentejas sin gluten,1.000000,0.333333,1.0


El Recall@10 medio sobre el catálogo completo es más bajo que el observado en la sección 3 sobre la
muestra de 1.500 productos. No es una contradicción: con diez veces más productos hay diez veces más
distractores plausibles compitiendo por las mismas diez posiciones, así que una caída de recall al pasar
de la muestra al catálogo completo es el comportamiento esperado, no un síntoma de que algo esté roto.

## 8. Fidelidad del ANN frente a un oráculo exacto

HNSW es un índice **aproximado**: puede perder algún vecino verdadero a cambio de responder mucho más
rápido que una búsqueda exhaustiva. Antes de culpar al modelo de embeddings por un mal resultado, hay que
descartar que el índice esté perdiendo candidatos que sí existen en el espacio vectorial.

El oráculo exacto se calcula con NumPy: un producto escalar de la consulta contra los 15.000 embeddings
del catálogo (ya en memoria desde la sección 4) y un `argsort` completo. Es deliberadamente la fuerza
bruta con la que se define "el vecino más cercano de verdad", sin ninguna aproximación de índice de por
medio. Se compara, para cada consulta de desarrollo, el top-10 exacto contra el top-10 que devuelve Chroma
y se mide qué fracción coincide (Recall@10 frente al oráculo). Esto es una medida de **fidelidad del
índice**, distinta de la relevancia medida en la sección 7: aquí no importa si el resultado le sirve al
usuario, solo si Chroma reprodujo fielmente la geometría que ya existía en los embeddings.

Esta comparación se hace ahora, con la colección recién ingerida y todavía sin las mutaciones de la
sección 11, para que el oráculo (calculado sobre `catalog_embeddings`) y el resultado de `search()`
describan exactamente el mismo estado del catálogo.

In [15]:
fidelity_rows = []
for _, query_row in dev_queries.iterrows():
    query_vector = encode_queries([query_row["query_text"]])[0]
    exact_scores = catalog_embeddings @ query_vector
    exact_top_positions = np.argsort(-exact_scores)[:TOP_K]
    exact_record_ids = set(catalog["record_id"].to_numpy()[exact_top_positions])

    ann_hits = search(query_row["query_text"], top_k=TOP_K)
    ann_record_ids = set(h["record_id"] for h in ann_hits)

    fidelity_rows.append({
        "query_id": query_row["query_id"],
        "recall_vs_oraculo@10": len(exact_record_ids & ann_record_ids) / TOP_K,
    })

fidelity_df = pd.DataFrame(fidelity_rows)
fidelity_recall = float(fidelity_df["recall_vs_oraculo@10"].mean())
print(f"Fidelidad media del ANN frente al oráculo exacto: {fidelity_recall:.4f}")
fidelity_df

Fidelidad media del ANN frente al oráculo exacto: 0.9875


,query_id,recall_vs_oraculo@10
0,13357,1.0
1,18868,1.0
2,28703,1.0
3,31224,0.9
4,33633,1.0
5,38249,1.0
6,43240,1.0
7,61533,1.0


Si esta fidelidad sale igual a 1.0 para una consulta, Chroma está reproduciendo exactamente el ranking de
fuerza bruta con la configuración HNSW elegida — cualquier resultado pobre de esa consulta en la sección 7
hay que atribuirlo entonces a la representación (el modelo o el texto codificado), no al índice. Un valor
por debajo de 1.0 apunta en cambio a que el índice aproximado está descartando algún candidato — la
sección siguiente retoma ambos casos con ejemplos concretos del propio catálogo.

## 9. Atribución de errores

Se examinan tres consultas de desarrollo concretas, cada una ilustrando una capa distinta del sistema, tal
como distingue el enunciado:

- **Representación**: el vecino exacto (oráculo) ya es semánticamente pobre — el fallo está en el modelo
  o en el texto codificado, no en el índice.
- **Índice**: el oráculo exacto recupera un producto que el ANN pierde — el fallo está en la
  aproximación de HNSW.
- **Datos o filtros**: falta información — aquí, cobertura incompleta de los juicios de relevancia — que
  hace que la métrica penalice resultados que podrían ser correctos.

Se eligen deliberadamente: la consulta con peor nDCG@10 de la sección 7 (`33633`), la consulta con menor
fidelidad ANN de la sección 8 (`31224`), y una consulta con muchísimos productos relevantes en los juicios
pero muy bajo Recall@10 (`38249`).

In [16]:
def show_oracle_top(query_id, top_n=5):
    """Muestra el top-n del oráculo exacto para una consulta de desarrollo, con su etiqueta
    de relevancia si el producto fue juzgado (y 'no juzgado' si no forma parte del pool de qrels)."""
    query_text = dev_queries.set_index("query_id").loc[query_id, "query_text"]
    judged = dev_qrels[dev_qrels["query_id"] == query_id].set_index("product_id")["esci_label"].to_dict()
    query_vector = encode_queries([query_text])[0]
    top_positions = np.argsort(-(catalog_embeddings @ query_vector))[:top_n]
    print(f'Consulta {query_id}: "{query_text}"')
    for position in top_positions:
        product_id = catalog.iloc[position]["product_id"]
        title = catalog.iloc[position]["title"][:70]
        label = judged.get(product_id, "no juzgado")
        print(f"  {product_id} [{label}] {title}")

print("nDCG@10 más bajo:", dict(dev_ranking_df.set_index("query_id").loc[33633, ["query_text", "ndcg@10"]]))
show_oracle_top(33633)

nDCG@10 más bajo: {'query_text': 'disfraz halloween talla grande hombre', 'ndcg@10': np.float64(0.0)}
Consulta 33633: "disfraz halloween talla grande hombre"
  B07TG2VB9F [no juzgado] LOLANTA Traje de Hombre de Hago de Oz para Niñas Traje de Hombre de Ho
  B07YG1C7D8 [no juzgado] Fossenfeliz Disfraces Medievales Mujer de Bruja Reina, Vestidos de Fie
  B07JCK13L4 [no juzgado] WIDMANN-Mad Joker Disfraz, multicolor, (XS) (WDM08039)
  B0753RX9VR [no juzgado] WIDMANN-Disfraz para adultos Malefica, large (03703)
  B09H42YXHC [no juzgado] HONGXUNJIE Halloween Mono de Ladrón de Banco para Carnaval,Disfraz de 


In [17]:
print("Recall@10 muy bajo con muchos relevantes disponibles:",
      dict(dev_ranking_df.set_index("query_id").loc[38249, ["query_text", "recall@10"]]))
show_oracle_top(38249)

Recall@10 muy bajo con muchos relevantes disponibles: {'query_text': 'estantes sin taladro habitacion', 'recall@10': np.float64(0.05714285714285714)}
Consulta 38249: "estantes sin taladro habitacion"
  1683258703 [no juzgado] La buena cocina sin sal
  B08KSL2NGZ [no juzgado] McFly - Nowhere Left To Run
  B098GX2DDJ [no juzgado] Máscaras sin nombre
  B07TJKBT7T [no juzgado] Lawless Darkness [Vinilo]
  B08BWTFMV4 [E] KES Estanteria Baño sin Taladro Baldas Pared sin Agujero con Orificio 


### Caso 1 — Representación: *"disfraz halloween talla grande hombre"* (query 33633, nDCG@10 = 0,00)

El oráculo exacto —el mismo espacio de embeddings que usa Chroma, sin ningún índice aproximado de por
medio— ya devuelve en su top-5 disfraces genéricos de Halloween que **no están en el conjunto de
productos juzgados** para esta consulta. La fidelidad ANN de esta consulta es 1.0 (sección 8): Chroma
reproduce fielmente ese mismo ranking, así que el índice no es el problema. El modelo capta bien el tema
general ("disfraz de Halloween") pero no pondera con fuerza suficiente las restricciones concretas de la
consulta —"talla grande" y "hombre"— frente a la enorme cantidad de disfraces de otro tipo que hay en el
catálogo. **Atribución: representación.** Un reranking léxico posterior que exigiera coincidencia de esos
términos, o un modelo con mejor sensibilidad a atributos, mitigaría este caso concreto.

### Caso 2 — Representación (negación): *"estantes sin taladro habitacion"* (query 38249, Recall@10 = 0,057
con 40 productos relevantes disponibles en los juicios)

Este es el caso más claro de los tres. El oráculo exacto sitúa en sus primeras posiciones un libro de
cocina ("La buena cocina **sin** sal"), un álbum ("Máscaras **sin** nombre") y una película ("Ángeles
**sin** alas") — todos comparten la palabra "sin" con la consulta, pero en un sentido completamente
distinto. El primer producto realmente relevante (una estantería de baño *sin* taladro, etiquetada `E`)
no aparece hasta la quinta posición. La fidelidad ANN de esta consulta también es 1.0, así que de nuevo el
índice reproduce fielmente el espacio — el problema nace en el encoder. **Atribución: representación.**
E5, como cualquier bi-encoder entrenado por similitud contrastiva, no tiene garantizado tratar una
negación como una restricción excluyente; aquí "sin" actúa casi como una palabra más de tema general en
vez de invertir el sentido de "taladro" — la misma limitación de los modelos densos frente a negaciones
que se discute en la asignatura, con un ejemplo real del catálogo.

In [18]:
query_31224_text = dev_queries.set_index("query_id").loc[31224, "query_text"]
query_vector_31224 = encode_queries([query_31224_text])[0]
exact_top10_31224 = set(catalog["record_id"].to_numpy()[np.argsort(-(catalog_embeddings @ query_vector_31224))[:10]])
ann_top10_31224 = set(h["record_id"] for h in search(query_31224_text, top_k=10))

only_in_oracle = exact_top10_31224 - ann_top10_31224
only_in_ann = ann_top10_31224 - exact_top10_31224
print(f'Consulta 31224: "{query_31224_text}" (fidelidad ANN = '
      f'{fidelity_df.set_index("query_id").loc[31224, "recall_vs_oraculo@10"]:.2f})')
for record_id in only_in_oracle:
    row = catalog[catalog["record_id"] == record_id].iloc[0]
    print("  Solo en el oráculo (lo pierde el ANN):", row["product_id"], row["title"][:70])
for record_id in only_in_ann:
    row = catalog[catalog["record_id"] == record_id].iloc[0]
    print("  Solo en Chroma (ocupa su lugar):     ", row["product_id"], row["title"][:70])

Consulta 31224: "cámaras bridge baratas" (fidelidad ANN = 0.90)
  Solo en el oráculo (lo pierde el ANN): 0130085316 Blackboard Premium Access Code Card
  Solo en Chroma (ocupa su lugar):      B001EHEH9I Bruder-S2412070 Mercedes-Benz peliculas y TV Camión Sprinter Caballo, 


### Caso 3 — Índice: *"cámaras bridge baratas"* (query 31224, fidelidad ANN = 0,9)

De los diez productos que devuelve el oráculo exacto, nueve coinciden con lo que devuelve Chroma; el
décimo puesto difiere. Al inspeccionar ambos IDs en la celda anterior, el producto que pierde el ANN y el
que aparece en su lugar son **los dos irrelevantes** para "cámaras bridge" (un código de acceso a una
plataforma educativa y un camión de juguete), con scores casi empatados. **Atribución: índice.** La
aproximación de HNSW se manifiesta exactamente donde cabía esperar: en el margen, entre candidatos con
puntuaciones muy próximas — no desplaza ninguno de los productos realmente relevantes que sí recupera el
oráculo en las primeras posiciones. Es la prueba de que una fidelidad de 0.9 no implica automáticamente un
10% peor de relevancia percibida por el usuario: aquí el efecto práctico es nulo.

### Resumen de la atribución

| Caso | Consulta | Capa | Evidencia clave |
|---|---|---|---|
| 1 | disfraz halloween talla grande hombre | Representación | El oráculo ya falla (nDCG=0); fidelidad ANN = 1.0 |
| 2 | estantes sin taladro habitación | Representación (negación) | El oráculo prioriza coincidencias léxicas con "sin" en sentido opuesto; fidelidad ANN = 1.0 |
| 3 | cámaras bridge baratas | Índice | Discrepancia oráculo/ANN solo en la posición 10, entre dos productos igualmente irrelevantes |

El patrón general es útil para decidir dónde intervenir: cuando la fidelidad ANN de una consulta es 1.0,
cualquier resultado deficiente hay que explicarlo por el modelo o los datos, no por la configuración de
Chroma. Solo cuando la fidelidad cae por debajo de 1.0 (como en el caso 3) tiene sentido tocar los
parámetros de `ef_search` antes que el modelo de embeddings — y, como muestra ese mismo caso, incluso
entonces conviene comprobar si el desacuerdo afecta a algún producto realmente relevante antes de asumir
que hace falta ajustar nada.

## 10. Latencia

Se mide la latencia de `search()` repitiendo las 8 consultas de desarrollo 20 veces (160 medidas en
total), con una consulta de calentamiento previa que no se contabiliza (la primera llamada suele incluir
inicializaciones que no representan el estado estable). Esto describe **esta ejecución concreta**: un
proceso Python local, en CPU, sin concurrencia ni llamadas de red a un servicio externo. No se usa para
comparar Chroma local con ningún proveedor cloud — como advierte el propio enunciado, ninguno de los dos
comparte hardware, red ni condiciones de carga, así que esa comparación no sería válida.

In [19]:
# Calentamiento: la primera consulta puede incluir inicializaciones puntuales.
search(dev_queries.iloc[0]["query_text"], top_k=TOP_K)

latency_samples_ms = []
for _ in range(20):
    for _, query_row in dev_queries.iterrows():
        start = time.perf_counter()
        search(query_row["query_text"], top_k=TOP_K)
        latency_samples_ms.append((time.perf_counter() - start) * 1000)

latency_p50_ms = float(np.percentile(latency_samples_ms, 50))
latency_p95_ms = float(np.percentile(latency_samples_ms, 95))
print(f"n={len(latency_samples_ms)} repeticiones | p50={latency_p50_ms:.1f} ms | p95={latency_p95_ms:.1f} ms")

n=160 repeticiones | p50=16.4 ms | p95=17.6 ms


## 11. Mutaciones del catálogo

`eventos_catalogo.csv` contiene 24 operaciones ordenadas por `sequence`: 8 actualizaciones (`UPSERT`
sobre productos ya existentes), 8 eliminaciones (`DELETE`) y 8 altas nuevas (`UPSERT` sobre productos que
no estaban en el catálogo). Se aplican **en ese orden** porque el propio fichero define el estado
esperado después de cada evento, no un conjunto de cambios independientes.

- Un `UPSERT` recalcula el embedding del texto actualizado y reemplaza el registro completo (vector +
  metadatos) bajo el mismo `record_id`.
- Un `DELETE` elimina el registro de la colección; no hace falta tocar ningún embedding.

El recuento total no debería cambiar: 8 bajas y 8 altas se compensan exactamente. Esa es la primera
comprobación, y es más informativa de lo que parece — si el número de registros cambiara, algo en el
proceso estaría mal (una fila mal clasificada, un ID duplicado, un evento repetido).

In [20]:
def apply_catalog_event(event_row):
    if event_row["operation"] == "DELETE":
        collection.delete(ids=[event_row["record_id"]])
    elif event_row["operation"] == "UPSERT":
        text = str(event_row["text"]) if pd.notna(event_row["text"]) else ""
        embedding = encode_passages([text])[0]
        collection.upsert(
            ids=[event_row["record_id"]],
            embeddings=[embedding.tolist()],
            documents=[text],
            metadatas=[{
                "product_id": str(event_row["product_id"]),
                "title": str(event_row["title"]) if pd.notna(event_row["title"]) else "",
                "brand": str(event_row["brand"]) if pd.notna(event_row["brand"]) else "",
                "color": str(event_row["color"]) if pd.notna(event_row["color"]) else "",
                "locale": str(event_row["locale"]),
                "catalog_version": int(event_row["catalog_version"]),
                "active": bool(event_row["active"]),
            }],
        )
    else:
        raise ValueError(f"Operación no reconocida: {event_row['operation']}")


count_before_events = collection.count()
for _, event_row in catalog_events.iterrows():
    apply_catalog_event(event_row)
count_after_events = collection.count()

print(f"Recuento antes: {count_before_events:,} | después de 24 eventos: {count_after_events:,}")
assert count_after_events == count_before_events, "8 altas y 8 bajas deberían compensarse exactamente"

Recuento antes: 15,000 | después de 24 eventos: 15,000


### Visibilidad por tipo de evento

Se comprueba una operación de cada tipo por dos rutas de lectura distintas: lectura directa por
`record_id` (`collection.get`) y búsqueda vectorial (`search`). Que una escritura se confirme no
significa todavía que sea visible por todas las rutas de lectura — comprobar ambas es lo que demuestra que
el sistema realmente actualizó lo que dice haber actualizado.

In [21]:
sample_upsert = catalog_events[catalog_events["operation"] == "UPSERT"].iloc[0]
by_id = collection.get(ids=[sample_upsert["record_id"]], include=["metadatas"])
visible_by_search = sample_upsert["record_id"] in [h["record_id"] for h in search(sample_upsert["title"], top_k=10)]
print("UPSERT · título tras el evento (por ID):", by_id["metadatas"][0]["title"])
print("UPSERT · visible también por búsqueda:", visible_by_search)
assert by_id["ids"] == [sample_upsert["record_id"]] and visible_by_search

UPSERT · título tras el evento (por ID): NIKE Legasee Legging Swoosh Pantalones Deportivos, Mujer, Negro (Black/White 011), S - ficha revisada
UPSERT · visible también por búsqueda: True


In [22]:
sample_delete = catalog_events[catalog_events["operation"] == "DELETE"].iloc[0]
by_id_deleted = collection.get(ids=[sample_delete["record_id"]])
print("DELETE · ausente por ID:", by_id_deleted["ids"] == [])
assert by_id_deleted["ids"] == []

DELETE · ausente por ID: True


In [23]:
sample_new = catalog_events[catalog_events["product_id"].str.startswith("AURUM-NEW")].iloc[0]
by_id_new = collection.get(ids=[sample_new["record_id"]], include=["metadatas"])
visible_new_by_search = sample_new["record_id"] in [h["record_id"] for h in search(sample_new["title"], top_k=10)]
print("Alta nueva · visible por ID:", by_id_new["ids"] == [sample_new["record_id"]])
print("Alta nueva · visible también por búsqueda:", visible_new_by_search)
assert by_id_new["ids"] == [sample_new["record_id"]] and visible_new_by_search

Alta nueva · visible por ID: True
Alta nueva · visible también por búsqueda: True


### Idempotencia de los eventos

El enunciado pide explícitamente que el proceso *"pueda ejecutarse una segunda vez sin cambiar el estado
final"*. Se reaplican los 24 eventos completos y se comprueba que el recuento no cambia: los `UPSERT`
vuelven a escribir el mismo estado sobre los mismos IDs y los `DELETE` sobre un registro ya ausente
simplemente no encuentran nada que borrar.

In [24]:
for _, event_row in catalog_events.iterrows():
    apply_catalog_event(event_row)

count_after_repeat = collection.count()
print(f"Recuento tras repetir los 24 eventos: {count_after_repeat:,}")
assert count_after_repeat == count_after_events, "Repetir los eventos no debería alterar el estado final"

Recuento tras repetir los 24 eventos: 15,000


## 12. Detección de altas duplicadas

### Idea general

Para cada ficha nueva se busca en la colección su vecino más cercano (la propia base vectorial es el
mecanismo que genera el candidato, tal como pide el enunciado) y se decide que es un duplicado si la
similitud con ese vecino supera un umbral. El candidato recuperado es siempre el `product_id` concreto
propuesto como duplicado, nunca solo un "sí/no".

### Un detalle de diseño: qué prefijo usar

Al codificar la ficha entrante se usa el prefijo `passage:`, igual que los productos del catálogo — **no**
`query:`. La razón es el papel que juega cada texto: aquí no se está expresando una necesidad breve para
buscar en un catálogo (eso es lo que representa `query:`), se están comparando **dos fichas de producto
entre sí** para ver si describen lo mismo. Es una comparación documento-documento, así que ambos lados
deben tratarse de forma simétrica.

### Calibración del umbral

`altas_desarrollo.csv` trae 14 casos etiquetados (7 duplicados reales, 7 altas genuinamente nuevas). Se
recupera el vecino más cercano de cada uno y se prueba un rango de umbrales de similitud, quedándose con
el que maximiza F1 **sobre estos 14 casos de desarrollo** — nunca mirando `altas_evaluacion.csv`, que no
tiene etiqueta visible. Fijar el umbral después de inspeccionar la evaluación invalidaría la medida.

Esta calibración se hace **después** de las mutaciones de la sección 11 a propósito: el catálogo contra el
que se comparan las altas de evaluación debe ser el catálogo ya actualizado, igual que ocurriría en
producción.

In [25]:
def nearest_catalog_candidate(text):
    """Devuelve (product_id, similitud) del vecino más cercano en el catálogo para un texto de ficha."""
    embedding = encode_passages([text])[0]
    response = collection.query(
        query_embeddings=[embedding.tolist()], n_results=1, include=["metadatas", "distances"]
    )
    if not response["ids"][0]:
        return None, 0.0
    metadata = response["metadatas"][0][0]
    distance = response["distances"][0][0]
    return metadata["product_id"], 1.0 - distance


dev_duplicate_rows = []
for _, incoming_row in dev_altas.iterrows():
    candidate_id, similarity = nearest_catalog_candidate(incoming_row["text"])
    dev_duplicate_rows.append({
        "incoming_id": incoming_row["incoming_id"],
        "candidate_product_id": candidate_id,
        "similarity": similarity,
        "is_duplicate": bool(incoming_row["is_duplicate"]),
        "reference_product_id": incoming_row["reference_product_id"],
    })
dev_duplicate_df = pd.DataFrame(dev_duplicate_rows)
dev_duplicate_df

,incoming_id,candidate_product_id,similarity,is_duplicate,reference_product_id
0,DEV-DUP-001,B000G3T55M,0.987419,True,B000G3T55M
1,DEV-DUP-002,B07PG68NDX,0.958204,True,B07NV4L2W5
2,DEV-DUP-003,B00BEFAR80,0.942250,True,B00BEFAR80
3,DEV-DUP-004,B076HKFZ8N,0.989658,True,B076HKFZ8N
4,DEV-DUP-005,B07JYHSK27,0.983738,True,B07JYHSK27
5,DEV-DUP-006,B07N379P73,0.929448,True,B07N379P73
6,DEV-DUP-007,B077FZDNJ2,0.974916,True,B077FZDNJ2
7,DEV-NEW-001,0241313333,0.884557,False,NaN
8,DEV-NEW-002,B0053JBKFW,0.877395,False,NaN
9,DEV-NEW-003,B06VVVQ72B,0.886418,False,NaN


Antes de fijar el umbral conviene comprobar que, en los casos etiquetados como duplicados, el candidato
recuperado por la base vectorial coincide con el producto de referencia. Si no coincidiera, el problema
estaría en la recuperación de candidatos, no en el umbral, y subir o bajar el umbral no lo arreglaría.

In [26]:
true_duplicates = dev_duplicate_df[dev_duplicate_df["is_duplicate"]]
candidate_matches_reference = (true_duplicates["candidate_product_id"] == true_duplicates["reference_product_id"])
print(f"Duplicados de desarrollo donde el vecino más cercano es el producto de referencia correcto: "
      f"{candidate_matches_reference.sum()}/{len(true_duplicates)}")
true_duplicates[~candidate_matches_reference]

Duplicados de desarrollo donde el vecino más cercano es el producto de referencia correcto: 6/7


,incoming_id,candidate_product_id,similarity,is_duplicate,reference_product_id
1,DEV-DUP-002,B07PG68NDX,0.958204,True,B07NV4L2W5


El único caso donde el candidato no coincide con la referencia (`DEV-DUP-002`) no es un fallo de
recuperación: el propio catálogo ya contiene dos fichas casi idénticas del mismo interruptor Meross
(`B07NV4L2W5` y `B07PG68NDX`), con el título diferenciado solo por una coma. Es justo el tipo de
contaminación de catálogo que describe el enunciado — productos publicados más de una vez —, así que
cuando llega una tercera ficha equivalente, la base vectorial acierta en identificarla como duplicado de
*alguno* de los dos gemelos existentes, aunque no sea literalmente el ID que el conjunto de desarrollo
señala como referencia. Para la decisión que importa aquí (¿hay que revisar esta alta antes de
publicarla?) el resultado es correcto; distinguir cuál de los dos gemelos es "el" original sería ya un
problema de limpieza del propio catálogo, no de detección de duplicados en la entrada.

In [27]:
threshold_results = []
for threshold in np.arange(0.80, 0.995, 0.005):
    predicted = dev_duplicate_df["similarity"] >= threshold
    true_positive = int((predicted & dev_duplicate_df["is_duplicate"]).sum())
    false_positive = int((predicted & ~dev_duplicate_df["is_duplicate"]).sum())
    false_negative = int((~predicted & dev_duplicate_df["is_duplicate"]).sum())
    precision = true_positive / (true_positive + false_positive) if (true_positive + false_positive) else 0.0
    recall = true_positive / (true_positive + false_negative) if (true_positive + false_negative) else 0.0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) else 0.0
    threshold_results.append({
        "threshold": round(float(threshold), 3), "precision": precision, "recall": recall, "f1": f1,
        "falsos_positivos": false_positive, "falsos_negativos": false_negative,
    })

threshold_df = pd.DataFrame(threshold_results)
best_threshold_row = threshold_df.loc[threshold_df["f1"].idxmax()]
DUPLICATE_THRESHOLD = float(best_threshold_row["threshold"])
print(f"Umbral elegido: {DUPLICATE_THRESHOLD} · precision={best_threshold_row['precision']:.3f} "
      f"recall={best_threshold_row['recall']:.3f} f1={best_threshold_row['f1']:.3f}")
threshold_df

Umbral elegido: 0.895 · precision=1.000 recall=1.000 f1=1.000


,threshold,precision,recall,f1,falsos_positivos,falsos_negativos
0,0.800,0.500000,1.000000,0.666667,7,0
1,0.805,0.500000,1.000000,0.666667,7,0
2,0.810,0.500000,1.000000,0.666667,7,0
3,0.815,0.500000,1.000000,0.666667,7,0
4,0.820,0.500000,1.000000,0.666667,7,0
5,0.825,0.500000,1.000000,0.666667,7,0
6,0.830,0.500000,1.000000,0.666667,7,0
7,0.835,0.500000,1.000000,0.666667,7,0
8,0.840,0.500000,1.000000,0.666667,7,0
9,0.845,0.500000,1.000000,0.666667,7,0


### Coste de cada tipo de error

Un **falso positivo** (marcar como duplicado una ficha realmente nueva) bloquea o retrasa la publicación
de un producto legítimo — un coste de fricción para el vendedor, pero reversible con una revisión manual.
Un **falso negativo** (dejar pasar un duplicado real) publica dos veces el mismo producto: degrada la
calidad del catálogo, divide reseñas y stock entre dos fichas, y ensucia cualquier métrica de búsqueda que
se calcule después. En un catálogo de marketplace, el falso negativo suele ser más caro de arreglar a
posteriori que el falso positivo, lo que justificaría — si hiciera falta mover el umbral — inclinarse
hacia un umbral algo más permisivo (menos falsos negativos) a costa de generar más revisiones manuales.

In [28]:
dev_predictions = dev_duplicate_df["similarity"] >= DUPLICATE_THRESHOLD
false_positive_cases = dev_duplicate_df[dev_predictions & ~dev_duplicate_df["is_duplicate"]]
false_negative_cases = dev_duplicate_df[~dev_predictions & dev_duplicate_df["is_duplicate"]]
print(f"Falsos positivos en desarrollo: {len(false_positive_cases)}")
print(f"Falsos negativos en desarrollo: {len(false_negative_cases)}")

Falsos positivos en desarrollo: 0
Falsos negativos en desarrollo: 0


### Aplicación al conjunto de evaluación

Se aplica exactamente la misma regla (mismo prefijo, mismo umbral) a las 14 altas de
`altas_evaluacion.csv`, sin ninguna etiqueta visible que pudiera sesgar la decisión.

In [29]:
eval_duplicate_rows = []
for _, incoming_row in eval_altas.iterrows():
    candidate_id, similarity = nearest_catalog_candidate(incoming_row["text"])
    predicted_duplicate = similarity >= DUPLICATE_THRESHOLD
    eval_duplicate_rows.append({
        "incoming_id": incoming_row["incoming_id"],
        "predicted_duplicate": bool(predicted_duplicate),
        "matched_product_id": candidate_id if predicted_duplicate else "",
        "score": similarity,
    })
eval_duplicate_df = pd.DataFrame(eval_duplicate_rows)
eval_duplicate_df

,incoming_id,predicted_duplicate,matched_product_id,score
0,EVAL-DUP-001,False,,0.876570
1,EVAL-DUP-002,True,B07H2Y8R6Y,0.907545
2,EVAL-DUP-003,False,,0.888576
3,EVAL-DUP-004,True,8417063064,0.910259
4,EVAL-DUP-005,False,,0.890417
5,EVAL-DUP-006,True,B01M6YTQAM,0.921846
6,EVAL-DUP-007,True,B01M0G3WP0,0.959817
7,EVAL-NEW-001,False,,0.883440
8,EVAL-NEW-002,False,,0.870856
9,EVAL-NEW-003,False,,0.878270


## 13. Artefactos de entrega

Se generan los tres artefactos que pide el enunciado, todos a partir de un único recorrido del notebook:

- `resultados_busqueda.csv`: top-10 para cada una de las 12 consultas ciegas de `consultas_evaluacion.csv`
  (sobre el catálogo ya actualizado por los eventos de la sección 11).
- `resultados_duplicados.csv`: la decisión ya calculada en la sección 12 para `altas_evaluacion.csv`.
- `metricas_desarrollo.json`: las métricas mínimas exigidas, calculadas sobre desarrollo.

In [30]:
search_result_rows = []
for _, query_row in eval_queries.iterrows():
    hits = search(query_row["query_text"], top_k=10)
    for hit in hits:
        search_result_rows.append({
            "evaluation_id": query_row["evaluation_id"],
            "rank": hit["rank"],
            "product_id": hit["product_id"],
            "score": hit["score"],
        })

search_results_df = pd.DataFrame(search_result_rows)

# Comprobación mínima antes de guardar: 10 IDs únicos por consulta.
ids_per_query = search_results_df.groupby("evaluation_id")["product_id"].nunique()
assert (ids_per_query == 10).all(), "Cada consulta ciega debe devolver 10 product_id únicos"

search_results_df.to_csv(RESULTS_DIR / "resultados_busqueda.csv", index=False)
print(f"Guardado resultados/resultados_busqueda.csv ({len(search_results_df)} filas)")
search_results_df.head()

Guardado resultados/resultados_busqueda.csv (120 filas)


,evaluation_id,rank,product_id,score
0,EVAL-100455-context,1,AURUM-NEW-001,0.905751
1,EVAL-100455-context,2,B07GSD93Q8,0.885659
2,EVAL-100455-context,3,B07C2TM76Y,0.885397
3,EVAL-100455-context,4,B00G7614BK,0.884092
4,EVAL-100455-context,5,B0071T3MOO,0.883851


In [31]:
eval_duplicate_df.to_csv(RESULTS_DIR / "resultados_duplicados.csv", index=False)
print(f"Guardado resultados/resultados_duplicados.csv ({len(eval_duplicate_df)} filas)")

Guardado resultados/resultados_duplicados.csv (14 filas)


In [32]:
duplicate_metrics_dev = {
    "duplicate_threshold": DUPLICATE_THRESHOLD,
    "duplicate_precision_dev": float(best_threshold_row["precision"]),
    "duplicate_recall_dev": float(best_threshold_row["recall"]),
    "duplicate_f1_dev": float(best_threshold_row["f1"]),
}

metrics_summary = {
    "ndcg_at_10": float(dev_metrics["ndcg@10"]),
    "recall_at_10": float(dev_metrics["recall@10"]),
    "mrr_at_10": float(dev_metrics["mrr@10"]),
    "latency_p50_ms": latency_p50_ms,
    "latency_p95_ms": latency_p95_ms,
    "fidelidad_ann_vs_oraculo_at_10": fidelity_recall,
    **duplicate_metrics_dev,
    "relevant_labels_for_recall_and_mrr": sorted(RELEVANT_LABELS),
    "modelo_embeddings": "intfloat/multilingual-e5-small",
    "metrica_indice": "cosine",
    "top_k": TOP_K,
}

with open(RESULTS_DIR / "metricas_desarrollo.json", "w", encoding="utf-8") as f:
    json.dump(metrics_summary, f, indent=2, ensure_ascii=False)

metrics_summary

{'ndcg_at_10': 0.533307015065813,
 'recall_at_10': 0.20505730828311475,
 'mrr_at_10': 0.7125,
 'latency_p50_ms': 16.415699999924982,
 'latency_p95_ms': 17.57804500007296,
 'fidelidad_ann_vs_oraculo_at_10': 0.9875,
 'duplicate_threshold': 0.895,
 'duplicate_precision_dev': 1.0,
 'duplicate_recall_dev': 1.0,
 'duplicate_f1_dev': 1.0,
 'relevant_labels_for_recall_and_mrr': ['E', 'S'],
 'modelo_embeddings': 'intfloat/multilingual-e5-small',
 'metrica_indice': 'cosine',
 'top_k': 10}

## 14. Limpieza (desactivada por defecto)

Este notebook solo crea recursos locales (una carpeta `chroma_db/` en el propio proyecto y una caché de
embeddings en `cache/`), así que no hay ningún coste cloud que el corrector pueda heredar ni ninguna
credencial involucrada. Aun así, se deja una celda de limpieza explícita y segura por defecto: solo borra
la colección si se cambia manualmente `CONFIRM_CLEANUP` a `True` en esta misma celda.

In [33]:
CONFIRM_CLEANUP = False  # cambiar explícitamente a True para borrar la colección local

if CONFIRM_CLEANUP:
    client.delete_collection(COLLECTION_NAME)
    print(f"Colección '{COLLECTION_NAME}' eliminada.")
else:
    print("Limpieza omitida (CONFIRM_CLEANUP=False). La colección y la caché de embeddings se conservan.")

Limpieza omitida (CONFIRM_CLEANUP=False). La colección y la caché de embeddings se conservan.


## 15. Conclusión y decisión recomendada para Aurum Market

**Qué se decidió y por qué:**

- **Representación**: E5 multilingüe (`intfloat/multilingual-e5-small`) sobre el campo `text` completo,
  con prefijos `query:`/`passage:` y normalización L2. Superó tanto al baseline léxico TF-IDF como a la
  variante que solo usaba el título, en las tres métricas de calidad medidas sobre desarrollo.
- **Base de datos vectorial**: Chroma local persistente, con HNSW (`cosine`, `max_neighbors=24`,
  `ef_construction=120`, `ef_search=128`) como único índice ANN disponible en su modo Single Node. La
  ingesta es idempotente por `record_id`, y el filtro de marca se resuelve dentro de la propia consulta
  vectorial.
- **Duplicados**: la propia base vectorial genera el candidato (vecino más cercano); un umbral de
  similitud, calibrado únicamente con los 14 casos de desarrollo, decide si se marca como duplicado.

**Qué muestra la atribución de errores**: sobre las consultas de desarrollo, la mayor parte de la pérdida
de calidad observada tiene origen en la **representación** (atributos poco ponderados, negaciones mal
capturadas), no en el índice — la fidelidad ANN media fue del 98,75%, y en el único caso con fidelidad
inferior a 1.0 el desacuerdo se producía entre dos candidatos igualmente irrelevantes. Eso concentra dónde
merece la pena invertir esfuerzo de mejora: en el modelo o el texto que se codifica, más que en ajustar el
índice.

**Qué cambiaría al crecer el catálogo:**

- Con varios millones de productos, `ef_search=128` y una consulta por producto en las mutaciones dejarían
  de ser triviales en latencia; habría que medir el compromiso latencia/recall con distintos valores de
  `ef_search` y, probablemente, mover la ingesta a un proceso por lotes asíncrono en vez de un bucle
  secuencial en el notebook.
- Chroma Single Node no está pensado para ese volumen con alta disponibilidad; el propio motor ofrece un
  modo distribuido que reparte el índice entre varios nodos, a costa de una operación más compleja
  (backups, autenticación, upgrades) que aquí no hizo falta considerar.
- El umbral de duplicados se calibró sobre solo 14 casos; con más histórico de altas sería razonable
  recalibrarlo (o incluso ajustarlo por categoría de producto) en vez de usar un único valor global.

**Lo que este notebook demuestra, y lo que no**: demuestra que, sobre datos reales con la suciedad
habitual de un catálogo (marcas ausentes, colores vacíos, títulos reordenados), un sistema denso simple
supera a la búsqueda literal, que el índice ANN elegido no pierde recall de forma apreciable frente a la
fuerza bruta, y que una regla de umbral sencilla, calibrada con honestidad sobre datos de desarrollo,
alcanza una detección de duplicados razonable. No demuestra qué proveedor cloud sería más rápido en
producción, ni sustituye una evaluación con usuarios reales sobre si el ranking resulta útil en la
práctica.